Urban Data Science & Smart Cities <br>
URSP688Y Spring 2026<br>
Instructor: Chester Harvey <br>
Urban Studies & Planning <br>
National Center for Smart Growth <br>
University of Maryland

# Demo 11 - Dashboards

Dashboards are custom user interfaces (UIs) for displaying and querying data, often in dynamic and interactive ways.

There are a few popular frameworks for creating dashboards with Python: [Dash](https://dash.plotly.com/), [Streamlit](https://streamlit.io/), and more recently [Shiny](https://shiny.posit.co/py/). All of these are essentially Python wrappers for creating web apps. Under the hood, they write HTML, CSS, and Javascript, which a web browser can render into an interactive web page.

While Python-based dashboard frameworks are convenient and, arguably, much more streamlined than writing web apps in raw HTML, CSS, and Javascript, they are still an immature technology that is evolving quickly. New tools that make it easier to stand up a functional dashboard are being introduced frequently by both private and open-source developers.

## Shiny Express
[Shiny](https://shiny.posit.co/) is a user-friendly dashboard framework initially developed for R that has more recently been released in a [Python version](https://shiny.posit.co/py/). It offers less flexibility than Dash, which has been the standard approach for Python-based dashboards, but can be easier for beginners. Even more recently, Posit, the company that develops Shiny, released [Shiny Express](https://shiny.posit.co/py/docs/express-in-depth.html), an even more streamlined approach for writing Shiny apps that compromises some flexibility for intuitive syntax.

Posit offers a convenient web interface, [shineylive.io](https://shinylive.io/py/examples/), for previewing and experimenting with Shiny Express code.

They also have a terrific [YouTube tutorial](https://www.youtube.com/watch?app=desktop&v=pBPF00M_bfU) introducing its basic features.

Here is the [full list of Shiny Express components](https://shiny.posit.co/py/api/express/).

## Let's Give Shiny Express a Try

Let's see if we can use Shiny Express to spin up our own web app visualizing current data from the [Capital Bikeshare API](https://gbfs.capitalbikeshare.com/gbfs/2.3/gbfs.json). The app will join data from available docked and dockless bikes with tract-level data [Equity Emphasis Areas](https://www.mwcog.org/maps/map-listing/equity-emphasis-areas-eeas/) from MWCOG and [neighborhood points](https://opendata.dc.gov/datasets/DCGIS::neighborhood-labels/explore) from the DC Open Data Portal. The goal is to visualize, in real time, the distribution of available bikes vis-a-vis disadvantaged communities within the city. Do bikes tend to be more available in more or less advantaged areas?

One we have developed the basic code for our app, we can copy it into a python file called `app.py`.

To run it in a browser, open a terminal window in the directory that contains that file and use this command:
`shiny run -r`.

We'll need to install a few new packages into our `688y` conda environment:
- `pip install plotly`
- `pip install shiny`
- `pip install shinywidgets`
- `pip install ipyleaflet`

In [ ]:
import demo11
import pandas as pd
import geopandas as gpd
import plotly.express as px
from ipyleaflet import GeoJSON, Map, CircleMarker, basemaps

%load_ext autoreload
%autoreload 2

In [ ]:
# Load Data
bikes_gdf, stations_gdf = demo11.load_bike_data()
bikes_gdf = pd.concat([
    bikes_gdf[['lat','lon','vehicle_type_id','timestamp','geometry']],
    demo11.expand_stations_into_bikes(stations_gdf)])
tracts_gdf = demo11.load_tracts()
eea_df = pd.read_csv('mwcog_eea_2016_2020.csv')
eea_df = demo11.clean_eea(eea_df)
nbhds_gdf = demo11.load_nbhds()

# Join equity emphasis area data to tracts
tracts_gdf = tracts_gdf.merge(eea_df, on='tract_id', how='left')

# Add tract data to nbhds with weighted average eea indices
# Also limit to nhbds with tract data
nbhds_gdf = demo11.add_tract_data_to_nbhds(nbhds_gdf, tracts_gdf)

# Attach neighborhoods to bikes
bikes_gdf = demo11.attach_points_to_points(bikes_gdf, nbhds_gdf)

In [ ]:
eea_color = 'blue'
other_color = 'red'

bikes_per_nbhd_df = demo11.count_bikes_per_nbhd(bikes_gdf).reset_index()

fig = px.bar(
    bikes_per_nbhd_df.head(20), 
    x='nbhd', 
    y='bikes', 
    color='eea',
    color_continuous_scale=[other_color, eea_color],
)
fig.update_layout(coloraxis_showscale=False)
fig.update_xaxes(tickangle=-90)

fig

In [ ]:
# Calculate midpoint for mapping
mid_lon, mid_lat = demo11.get_gdf_midpoint(nbhds_gdf)

bikes_per_nbhd_df = demo11.count_bikes_per_nbhd(bikes_gdf)
nbhds_with_bike_counts_gdf = demo11.attach_counts_to_nbhd_points(bikes_per_nbhd_df, nbhds_gdf) 

# Build map
m = Map(
    basemap=basemaps.CartoDB.Positron,
    center=(mid_lat, mid_lon), 
    zoom=12,
    scroll_wheel_zoom=True,
)  

multiplier = 0.2
for nbhd in nbhds_with_bike_counts_gdf.itertuples():
    circle_marker = CircleMarker()
    lat = nbhd.geometry.y
    lon = nbhd.geometry.x
    circle_marker.location = (lat, lon)
    radius = demo11.int_at_least_one(nbhd.bikes * multiplier)
    circle_marker.radius = radius
    if nbhd.eea == 1:
        circle_marker.color = eea_color
    else:
        circle_marker.color = other_color
    circle_marker.fill_color = "#ffffff00"
    circle_marker.weight=1
    m.add(circle_marker)

m